# Belka · 01 · Bronze
Carga incremental dos arquivos enviados ao Volume `workspace.belka.chegada`.

In [0]:
%sql
-- Preparação: schema, Volume e tabelas Bronze.
CREATE SCHEMA IF NOT EXISTS workspace.belka;
CREATE VOLUME IF NOT EXISTS workspace.belka.chegada;
CREATE TABLE IF NOT EXISTS workspace.belka.bronze_clima
  (dia_coleta STRING, arquivo_origem STRING, carregado_em TIMESTAMP);
CREATE TABLE IF NOT EXISTS workspace.belka.bronze_populacao
  (dia_coleta STRING, arquivo_origem STRING, carregado_em TIMESTAMP);

## Contagem ANTES da carga

In [0]:
%sql
SELECT 'bronze_clima' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT arquivo_origem) AS arquivos,
       MIN(dia_coleta) AS primeiro_dia, MAX(dia_coleta) AS ultimo_dia
FROM workspace.belka.bronze_clima
UNION ALL
SELECT 'bronze_populacao', COUNT(*), COUNT(DISTINCT arquivo_origem),
       MIN(dia_coleta), MAX(dia_coleta)
FROM workspace.belka.bronze_populacao;

tabela,linhas,arquivos,primeiro_dia,ultimo_dia
bronze_clima,65,13,2026-10-25,2026-11-06
bronze_populacao,13,13,2026-10-25,2026-11-06


## Carga: clima (Open-Meteo)

In [0]:
%sql
COPY INTO workspace.belka.bronze_clima
FROM (
  SELECT
    *,
    _metadata.file_path  AS arquivo_origem,
    current_timestamp()  AS carregado_em
  FROM '/Volumes/workspace/belka/chegada/clima/'
)
FILEFORMAT = JSON
PATTERN = 'clima_*.json'
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


## Carga: população (IBGE)

In [0]:
%sql
COPY INTO workspace.belka.bronze_populacao
FROM (
  SELECT
    *,
    _metadata.file_path  AS arquivo_origem,
    current_timestamp()  AS carregado_em
  FROM '/Volumes/workspace/belka/chegada/ibge/'
)
FILEFORMAT = JSON
PATTERN = 'ibge_populacao_*.json'
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


## Contagem DEPOIS da carga

In [0]:
%sql
SELECT 'bronze_clima' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT arquivo_origem) AS arquivos,
       MIN(dia_coleta) AS primeiro_dia, MAX(dia_coleta) AS ultimo_dia
FROM workspace.belka.bronze_clima
UNION ALL
SELECT 'bronze_populacao', COUNT(*), COUNT(DISTINCT arquivo_origem),
       MIN(dia_coleta), MAX(dia_coleta)
FROM workspace.belka.bronze_populacao;

tabela,linhas,arquivos,primeiro_dia,ultimo_dia
bronze_clima,65,13,2026-10-25,2026-11-06
bronze_populacao,13,13,2026-10-25,2026-11-06


## Lotes carregados

In [0]:
%sql
SELECT 'bronze_clima' AS tabela, carregado_em, COUNT(DISTINCT arquivo_origem) AS arquivos,
       COUNT(*) AS linhas, MIN(dia_coleta) AS de, MAX(dia_coleta) AS ate
FROM workspace.belka.bronze_clima GROUP BY carregado_em
UNION ALL
SELECT 'bronze_populacao', carregado_em, COUNT(DISTINCT arquivo_origem),
       COUNT(*), MIN(dia_coleta), MAX(dia_coleta)
FROM workspace.belka.bronze_populacao GROUP BY carregado_em
ORDER BY tabela, carregado_em;

tabela,carregado_em,arquivos,linhas,de,ate
bronze_clima,2026-11-07T00:03:05.809Z,7,35,2026-10-25,2026-10-31
bronze_clima,2026-11-07T00:08:19.069Z,6,30,2026-11-01,2026-11-06
bronze_populacao,2026-11-07T00:03:10.536Z,7,7,2026-10-25,2026-10-31
bronze_populacao,2026-11-07T00:08:24.951Z,6,6,2026-11-01,2026-11-06


## Prova de que não há duplicação

In [0]:
%sql
-- Clima: nenhum município aparece duas vezes no mesmo dia
SELECT municipio_id, dia_coleta, COUNT(*) AS vezes
FROM workspace.belka.bronze_clima
GROUP BY municipio_id, dia_coleta
HAVING COUNT(*) > 1;

municipio_id,dia_coleta,vezes


In [0]:
%sql
-- População: um registro por dia de coleta
SELECT dia_coleta, COUNT(*) AS vezes
FROM workspace.belka.bronze_populacao
GROUP BY dia_coleta
HAVING COUNT(*) > 1;

dia_coleta,vezes


## Histórico das cargas

In [0]:
%sql
DESCRIBE HISTORY workspace.belka.bronze_clima;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-07T00:08:21.000Z,76186718158206,fernandakaory2@gmail.com,COPY INTO,"Map(statsOnLoad -> true, clusterBy -> [], clusteringOnWriteStatus -> null)",null,List(2781832021275861),75c18ad8-08dc-41d0-8ba3-e06119b919f0,1106-233146-zxhjd9rh-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 30, numOutputBytes -> 12667, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-07T00:03:08.000Z,76186718158206,fernandakaory2@gmail.com,COPY INTO,"Map(statsOnLoad -> true, clusterBy -> [], clusteringOnWriteStatus -> null)",null,List(2781832021275861),f1c5cfe9-752a-44c0-8da8-08df8e199354,1106-233146-zxhjd9rh-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 35, numOutputBytes -> 13363, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-07T00:02:49.000Z,76186718158206,fernandakaory2@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""09644021-3d5c-4f15-ab4a-bfc8aa3ec6d1"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f66b2858-2079-4004-868c-7e2f2167779a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-21409d01-6ea4-46f4-9fc1-a783ecb6084a""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(2781832021275861),1314bab6-2b10-47f9-bd32-e1e0fcfea0ce,1106-233146-zxhjd9rh-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
DESCRIBE TABLE workspace.belka.bronze_clima;

col_name,data_type,comment
dia_coleta,string,null
arquivo_origem,string,null
carregado_em,timestamp,null
coletado_em,string,null
elevation,double,null
fonte,string,null
generationtime_ms,double,null
hourly,"struct,relative_humidity_2m:array,temperature_2m:array,time:array,wind_gusts_10m:array>",null
hourly_units,struct,null
latitude,double,null


In [0]:
DESCRIBE TABLE workspace.belka.bronze_populacao;

col_name,data_type,comment
dia_coleta,string,null
arquivo_origem,string,null
carregado_em,timestamp,null
coletado_em,string,null
fonte,string,null
id,string,null
resultados,"array,series:array,nome:string>,serie:struct<2026:string>>>>>",null
unidade,string,null
variavel,string,null
# Clustering Information,,


In [0]:
SELECT *
FROM workspace.belka.bronze_clima
ORDER BY dia_coleta, municipio_id
LIMIT 10;

dia_coleta,arquivo_origem,carregado_em,coletado_em,elevation,fonte,generationtime_ms,hourly,hourly_units,latitude,location_id,longitude,municipio_id,municipio_nome,timezone,timezone_abbreviation,uf,utc_offset_seconds
2026-10-25,dbfs:/Volumes/workspace/belka/chegada/clima/clima_2026-10-25.json,2026-11-07T00:03:05.809Z,2026-11-06T23:02:39.353418Z,693.0,open-meteo,0.0654458999633789,"List(List(3.5, 0.8, 1.3, 1.0, 0.9, 4.7, 7.7, 5.2, 7.0, 5.6, 1.3, 0.4, 0.4, 0.3, 0.3, 0.2, 2.7, 0.4, 0.1, 0.0, 0.1, 0.0, 0.0, 0.0), List(96, 96, 96, 96, 96, 96, 96, 97, 97, 93, 96, 93, 84, 84, 78, 87, 93, 91, 95, 95, 95, 96, 95, 95), List(20.6, 20.5, 20.7, 19.8, 19.8, 19.7, 19.7, 19.8, 20.2, 20.8, 20.9, 21.6, 23.0, 22.8, 23.7, 21.7, 20.9, 21.2, 20.7, 20.2, 19.9, 19.1, 18.2, 17.9), List(2026-10-25T00:00, 2026-10-25T01:00, 2026-10-25T02:00, 2026-10-25T03:00, 2026-10-25T04:00, 2026-10-25T05:00, 2026-10-25T06:00, 2026-10-25T07:00, 2026-10-25T08:00, 2026-10-25T09:00, 2026-10-25T10:00, 2026-10-25T11:00, 2026-10-25T12:00, 2026-10-25T13:00, 2026-10-25T14:00, 2026-10-25T15:00, 2026-10-25T16:00, 2026-10-25T17:00, 2026-10-25T18:00, 2026-10-25T19:00, 2026-10-25T20:00, 2026-10-25T21:00, 2026-10-25T22:00, 2026-10-25T23:00), List(23.8, 18.4, 39.6, 43.6, 24.1, 30.2, 33.8, 27.4, 33.8, 30.2, 34.9, 34.6, 32.8, 33.1, 28.1, 25.9, 28.8, 15.1, 13.3, 2.5, 7.6, 15.5, 24.5, 22.0))","List(mm, %, °C, iso8601, km/h)",-22.952549,3,-47.066803,3509502,Campinas,America/Sao_Paulo,GMT-3,SP,-10800
2026-10-25,dbfs:/Volumes/workspace/belka/chegada/clima/clima_2026-10-25.json,2026-11-07T00:03:05.809Z,2026-11-06T23:02:39.353418Z,759.0,open-meteo,0.04601478576660156,"List(List(3.7, 2.0, 1.5, 2.4, 1.1, 3.9, 2.8, 3.3, 6.4, 1.7, 0.4, 0.8, 0.5, 0.2, 0.4, 0.5, 0.6, 0.3, 0.2, 0.8, 0.3, 0.0, 0.0, 0.0), List(97, 99, 98, 99, 97, 98, 98, 98, 98, 98, 94, 92, 89, 79, 78, 95, 95, 94, 97, 98, 98, 99, 99, 98), List(19.1, 18.8, 18.7, 18.2, 18.3, 18.3, 18.2, 18.5, 18.9, 19.5, 20.6, 20.8, 21.2, 22.9, 23.2, 19.6, 19.6, 19.6, 19.0, 18.3, 17.6, 16.9, 16.3, 16.2), List(2026-10-25T00:00, 2026-10-25T01:00, 2026-10-25T02:00, 2026-10-25T03:00, 2026-10-25T04:00, 2026-10-25T05:00, 2026-10-25T06:00, 2026-10-25T07:00, 2026-10-25T08:00, 2026-10-25T09:00, 2026-10-25T10:00, 2026-10-25T11:00, 2026-10-25T12:00, 2026-10-25T13:00, 2026-10-25T14:00, 2026-10-25T15:00, 2026-10-25T16:00, 2026-10-25T17:00, 2026-10-25T18:00, 2026-10-25T19:00, 2026-10-25T20:00, 2026-10-25T21:00, 2026-10-25T22:00, 2026-10-25T23:00), List(21.6, 20.5, 34.2, 47.2, 25.9, 23.8, 31.0, 32.0, 11.5, 18.4, 28.4, 32.4, 32.8, 34.2, 33.8, 28.4, 19.4, 16.6, 16.9, 18.7, 16.9, 19.8, 13.0, 15.1))","List(mm, %, °C, iso8601, km/h)",-23.304043,2,-46.74713,3516408,Franco da Rocha,America/Sao_Paulo,GMT-3,SP,-10800
2026-10-25,dbfs:/Volumes/workspace/belka/chegada/clima/clima_2026-10-25.json,2026-11-07T00:03:05.809Z,2026-11-06T23:02:39.353418Z,740.0,open-meteo,0.029802322387695312,"List(List(4.3, 3.4, 1.3, 2.3, 1.4, 2.3, 0.7, 4.6, 5.3, 1.3, 0.2, 0.5, 1.2, 1.3, 1.4, 1.7, 1.4, 0.4, 0.2, 0.2, 0.1, 0.0, 0.0, 0.1), List(98, 99, 97, 97, 99, 100, 99, 99, 98, 96, 90, 89, 90, 88, 89, 94, 93, 92, 93, 94, 94, 96, 97, 96), List(19.1, 19.0, 19.0, 18.8, 18.5, 18.5, 18.4, 18.7, 19.2, 20.2, 20.9, 20.8, 20.8, 21.0, 20.9, 19.5, 19.5, 19.1, 18.1, 17.5, 17.2, 16.7, 16.1, 16.2), List(2026-10-25T00:00, 2026-10-25T01:00, 2026-10-25T02:00, 2026-10-25T03:00, 2026-10-25T04:00, 2026-10-25T05:00, 2026-10-25T06:00, 2026-10-25T07:00, 2026-10-25T08:00, 2026-10-25T09:00, 2026-10-25T10:00, 2026-10-25T11:00, 2026-10-25T12:00, 2026-10-25T13:00, 2026-10-25T14:00, 2026-10-25T15:00, 2026-10-25T16:00, 2026-10-25T17:00, 2026-10-25T18:00, 2026-10-25T19:00, 2026-10-25T20:00, 2026-10-25T21:00, 2026-10-25T22:00, 2026-10-25T23:00), List(24.5, 21.2, 20.9, 37.4, 18.0, 18.7, 12.6, 13.3, 27.0, 29.5, 34.6, 34.9, 27.7, 24.5, 25.9, 24.5, 18.0, 18.7, 21.2, 20.9, 17.3, 14.8, 16.6, 12.6))","List(mm, %, °C, iso8601, km/h)",-23.44464,1,-46.561523,3518800,Guarulhos,America/Sao_Paulo,GMT-3,SP,-10800
2026-10-25,dbfs:/Volumes/workspace

In [0]:
SELECT *
FROM workspace.belka.bronze_populacao
ORDER BY dia_coleta
LIMIT 10;

dia_coleta,arquivo_origem,carregado_em,coletado_em,fonte,id,resultados,unidade,variavel
2026-10-25,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-25.json,2026-11-07T00:03:10.536Z,2026-11-06T23:02:38.644272Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(1351284)), List(List(3516408, List(N6, Município), Franco da Rocha (SP)), List(150648)), List(List(3509502, List(N6, Município), Campinas (SP)), List(1189761)), List(List(3550704, List(N6, Município), São Sebastião (SP)), List(84514)))))",Pessoas,População residente estimada
2026-10-26,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-26.json,2026-11-07T00:03:10.536Z,2026-11-06T23:02:34.748113Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(1351284)), List(List(3516408, List(N6, Município), Franco da Rocha (SP)), List(150648)), List(List(3509502, List(N6, Município), Campinas (SP)), List(1189761)), List(List(3550704, List(N6, Município), São Sebastião (SP)), List(84514)))))",Pessoas,População residente estimada
2026-10-27,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-27.json,2026-11-07T00:03:10.536Z,2026-11-06T23:02:43.504258Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(1351284)), List(List(3516408, List(N6, Município), Franco da Rocha (SP)), List(150648)), List(List(3509502, List(N6, Município), Campinas (SP)), List(1189761)), List(List(3550704, List(N6, Município), São Sebastião (SP)), List(84514)))))",Pessoas,População residente estimada
2026-10-28,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-28.json,2026-11-07T00:03:10.536Z,2026-11-06T23:02:48.886266Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(1351284)), List(List(3516408, List(N6, Município), Franco da Rocha (SP)), List(150648)), List(List(3509502, List(N6, Município), Campinas (SP)), List(1189761)), List(List(3550704, List(N6, Município), São Sebastião (SP)), List(84514)))))",Pessoas,População residente estimada
2026-10-29,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-29.json,2026-11-07T00:03:10.536Z,2026-11-06T23:02:52.667765Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(1351284)), List(List(3516408, List(N6, Município), Franco da Rocha (SP)), List(150648)), List(List(3509502, List(N6, Município), Campinas (SP)), List(1189761)), List(List(3550704, List(N6, Município), São Sebastião (SP)), List(84514)))))",Pessoas,População residente estimada
2026-10-30,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-30.json,2026-11-07T00:03:10.536Z,2026-11-06T23:02:56.716111Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(1351284)), List(List(3516408, List(N6, Município), Franco da Rocha (SP)), List(150648)), List(List(3509502, List(N6, Município), Campinas (SP)), List(1189761)), List(List(3550704, List(N6, Município), São Sebastião (SP)), List(84514)))))",Pessoas,População residente estimada
2026-10-31,dbfs:/Volumes/workspace/belka/chegada/ibge/ibge_populacao_2026-10-31.json,2026-11-07T00:03:10.536Z,2026-11-06T23:05:20.830328Z,ibge-agregado-6579,9324,"List(List(List(), List(List(List(3550308, List(N6, Município), São Paulo (SP)), List(11911337)), List(List(3518800, List(N6, Município), Guarulhos (SP)), List(13